In [ ]:
# Typical steps
# 1. Application parses policy documents and extracts relevant information to create a knowledge base.

# This data will be stored in vector database and will be used as context for question answering.
content_corpus = [
        "Employees are compensated on a bi-weekly basis through direct deposit.",       
        "Employees must submit a leave request for approval.", 
        "Company internet must be used for work-related tasks only.",
        "Company internet is a broadband internet.",
        "Employees can take an hour break.",
        "Interact with each employee with Respect"
]


In [ ]:
# Goal: Retrieve relevant documents based on user query (a.k.a. Context Retrieval)

%pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(content_corpus)

In [ ]:
doc_vectors

In [ ]:
# Step 3: User Query and Semantic Matching

query = "What’s the internet policy?"
query_vec = model.encode([query])[0]
query_vec

In [ ]:
similarities = model.similarity(query_vec, doc_vectors)
print(type(similarities))

# Ensure it's a 1D numpy array
import numpy as np
similarities = np.asarray(similarities).squeeze()
similarities

In [ ]:

# Now get top 3 indices based on similarity scores
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_scores

In [ ]:
top_scores

In [ ]:
documents = content_corpus
top_docs = [documents[i] for i in top_3_indices]


print (top_docs)
context = ", ".join(top_docs)
context

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_ai(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create
    print(f"Context: {context}")
    print(f"User Question: {query}")    
    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        # messages=[
        #     {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
        #     {"role": "user", "content": prompt}
        # ]
        messages=[
            {"role": "system", "content": '''
             You are an assistant who answers only based on the given context.
             '''},
            {"role": "user", "content": f"Context: {context}\n\n User Question: {query}"} 
        ]

    )
    return llm_response.choices[0].message.content  


In [ ]:
print(f"User query: {query}")
print(f"Context: {context}")

print(f"\n\nOpen AI Response: {response}")

In [ ]:
#Reference Data
expected_answer = "The company's internet must be used for work-related tasks only."    
actual_response = response

# Call ChatCompltions API to compare expected vs actual
#LLM AS A JUDGE
comparison_prompt = f"""
You are an expert evaluator. Compare the actual response to the expected answer and determine if they match in meaning.
Expected Answer: {expected_answer}
Actual Response: {actual_response}
Do they match in meaning? Answer with 'Yes' or 'No' and provide a brief explanation.
"""
comparison_response = my_client.chat.completions.create(
    model="gpt-5-nano",     
    messages=[
        {"role": "system", "content": "You are an expert evaluator."},
        {"role": "user", "content": comparison_prompt}
    ]
)   
print(f"\nExpected Answer: {comparison_response.choices[0].message.content}")

In [ ]:
print (f"\nExpected Answer: {expected_answer}")
print (f"\nActual Response: {actual_response}")